# SkinAI — DS14 + DermNet 혼합 학습 (DS14_mixed)

**목적**: AI Hub 합성 데이터(9,600장) + DermNet NZ 실제 임상 이미지(3,373장) 혼합 학습으로  
합성 → 실제 이미지 도메인 갭 완화 (목표: 실제 이미지 Top-1 ≥ 75%)

| 데이터 | 이미지 수 | 클래스 | 용도 |
|--------|----------|--------|------|
| AI Hub 08-14 합성 | 9,600 | 6종 전체 | 학습 |
| DermNet NZ (train) | 3,373 | 건선·아토피·여드름 3종 | 학습 |
| DermNet NZ (test) | 1,096 | 건선·아토피·여드름 3종 | **홀드아웃 평가만** |

> ⚠️ **주사·지루피부염·정상** 클래스는 DermNet에 없으므로 AI Hub 합성 데이터만으로 학습됩니다.

In [3]:
# GPU / RAM 확인
!nvidia-smi
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f"시스템 RAM: {ram_gb:.1f} GB")

In [ ]:
# ── 셀 1: 환경 감지 ────────────────────────────────────────────
import os
from pathlib import Path

try:
    import google.colab
    IS_COLAB = True
    COLAB_ROOT = "/content/colab_skin_ai"
    PROJECT_ROOT = COLAB_ROOT
except ImportError:
    IS_COLAB = False
    PROJECT_ROOT = str(Path.cwd())

print(f"환경        : {'Google Colab' if IS_COLAB else '로컬'}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

In [ ]:
# ── 셀 2: Drive 마운트 (Colab 전용) ────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/skin_ai"
else:
    print("로컬 환경 — Drive 마운트 건너뜀")

In [ ]:
# ── 셀 3: 소스코드 clone / pull (Colab 전용) ────────────────────
if IS_COLAB:
    from dotenv import load_dotenv

    _env_path = f"{DRIVE_ROOT}/.env"
    if Path(_env_path).exists():
        load_dotenv(_env_path)
    else:
        print(f"경고: {_env_path} 없음 — GITHUB_TOKEN 없이 시도합니다")

    _token = os.getenv("GITHUB_TOKEN", "")
    _repo_url = (
        f"https://{_token}@github.com/kyoe-23/skin_ai.git"
        if _token else
        "https://github.com/kyoe-23/skin_ai.git"
    )

    if not Path(COLAB_ROOT).exists():
        !git clone {_repo_url} {COLAB_ROOT}
    else:
        !git -C {COLAB_ROOT} pull
else:
    print("로컬 환경 — 클론 건너뜀")

In [ ]:
# ── 셀 4: 프로젝트 루트 이동 + 데이터 심링크 설정 ────────────────
os.chdir(PROJECT_ROOT)
print(f"현재 디렉토리: {os.getcwd()}")

if IS_COLAB:
    os.makedirs("data/processed", exist_ok=True)

    # AI Hub DS14 원본 데이터
    DRIVE_DS14 = f"{DRIVE_ROOT}/data/dataset_14"
    if Path(DRIVE_DS14).exists():
        !ln -sfn {DRIVE_DS14} data/dataset_14
        print("data/dataset_14 심링크 완료")
    else:
        print(f"❌ 경고: Drive에 dataset_14 없음 — {DRIVE_DS14}")

    # DermNet 실제 임상 이미지
    DRIVE_DERMNET = f"{DRIVE_ROOT}/data/dermnet"
    if Path(DRIVE_DERMNET).exists():
        !ln -sfn {DRIVE_DERMNET} data/dermnet
        print("data/dermnet 심링크 완료")
    else:
        print(f"❌ 경고: Drive에 dermnet 없음 — {DRIVE_DERMNET}")
        print("  → Drive에 data/dermnet/ 디렉토리를 업로드한 후 다시 실행하세요.")

    # DS14 전처리 CSV (이미 Drive에 있으면 심링크)
    DRIVE_DS14_CSV = f"{DRIVE_ROOT}/data/processed/DS14"
    if Path(DRIVE_DS14_CSV).exists():
        !ln -sfn {DRIVE_DS14_CSV} data/processed/DS14
        print("data/processed/DS14 심링크 완료")
    else:
        print(f"❌ 경고: DS14 전처리 CSV 없음 — {DRIVE_DS14_CSV}")
        print("  → ai/preprocessing/aihub_preprocessor.py를 먼저 실행하세요.")

    # DermNet 전처리 CSV (이미 Drive에 있으면 심링크 — 셀 7 건너뜀 가능)
    DRIVE_EXT_CSV = f"{DRIVE_ROOT}/data/processed/dermnet"
    if Path(DRIVE_EXT_CSV).exists():
        !ln -sfn {DRIVE_EXT_CSV} data/processed/dermnet
        print("data/processed/dermnet 심링크 완료 (셀 7 건너뛸 수 있습니다)")
    else:
        print("⚠️  DermNet 전처리 CSV 없음 — 셀 7 실행 후 자동 생성됩니다")

else:
    for d in ["data/dataset_14", "data/dermnet", "data/processed/DS14", "data/processed/dermnet"]:
        status = "✅" if Path(d).exists() else "❌"
        print(f"{status} {d}")

!ls data/

In [ ]:
# ── 셀 5: 패키지 설치 ───────────────────────────────────────────
!pip install -q \
    torch torchvision \
    pandas pillow tqdm \
    matplotlib python-dotenv scikit-learn

In [ ]:
# ── 셀 6: 학습 전 체크리스트 ────────────────────────────────────
import torch
import pandas as pd
from pathlib import Path

checks = []
warnings = []

# 1. GPU
gpu_ok = torch.cuda.is_available()
checks.append(("GPU 사용 가능", gpu_ok))
if gpu_ok:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {name} ({vram:.0f} GB)")

# 2. DS14 전처리 CSV
ds14_train = Path("data/processed/DS14/train.csv")
ds14_val   = Path("data/processed/DS14/val.csv")
ds14_ok = ds14_train.exists() and ds14_val.exists()
checks.append(("DS14 전처리 CSV (train/val)", ds14_ok))
if ds14_ok:
    n = len(pd.read_csv(ds14_train)) + len(pd.read_csv(ds14_val))
    print(f"  DS14 총 {n:,}건")

# 3. DermNet 원본 데이터
dermnet_train = Path("data/dermnet/train")
dermnet_test  = Path("data/dermnet/test")
dermnet_ok = dermnet_train.exists() and dermnet_test.exists()
checks.append(("DermNet 원본 데이터 (train/test)", dermnet_ok))
if dermnet_ok:
    n_train = sum(1 for f in dermnet_train.rglob("*") if f.is_file())
    n_test  = sum(1 for f in dermnet_test.rglob("*") if f.is_file())
    print(f"  DermNet train {n_train:,}장 / test {n_test:,}장")

# 4. DermNet 전처리 CSV (없으면 다음 셀에서 생성)
ext_train = Path("data/processed/dermnet/train.csv")
ext_csv_ok = ext_train.exists()
checks.append(("DermNet 전처리 CSV (data/processed/dermnet)", ext_csv_ok))
if not ext_csv_ok:
    warnings.append("  → 셀 7(외부 데이터 전처리)을 실행하면 자동 생성됩니다.")

# 5. 클래스 커버리지 경고
warnings.append("  ⚠️  DermNet에 '주사·지루피부염·정상' 클래스가 없음 — AI Hub 데이터만으로 학습됩니다.")
warnings.append("  ⚠️  홀드아웃 평가(data/processed/dermnet/test.csv)는 건선·아토피·여드름 3종만 측정됩니다.")

# ── 출력 ──────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("학습 전 체크리스트")
print("=" * 50)
all_ok = True
for name, ok in checks:
    status = "✅" if ok else "❌"
    print(f"{status} {name}")
    if not ok:
        all_ok = False

print("\n주의사항:")
for w in warnings:
    print(w)

print("\n" + ("🟢 모든 필수 조건 충족 — 학습 진행 가능" if all_ok else "🔴 위 항목을 먼저 해결하세요"))

In [ ]:
# ── 셀 7: 외부 데이터 전처리 (DermNet → CSV) ─────────────────────
# Drive의 기존 CSV가 절대경로이면 자동 재생성 (Colab 환경 경로 불일치 방지)
import json as _json
import pandas as _pd
from pathlib import Path

EXT_CSV_DIR = "data/processed/dermnet"
train_csv = Path(f"{EXT_CSV_DIR}/train.csv")

needs_regen = True
if train_csv.exists():
    _sample = _pd.read_csv(train_csv, nrows=1)
    if _sample["image_path"].iloc[0].startswith("/"):
        print("⚠️  절대경로 CSV 감지 — Colab 경로로 재생성합니다")
    else:
        needs_regen = False
        print(f"✅ 이미 존재 (상대경로) — {EXT_CSV_DIR}/train.csv")

if needs_regen:
    print("DermNet 전처리 시작...")
    !python -m ai.preprocessing.external_preprocessor \
        --root_dir data/dermnet \
        --output_dir {EXT_CSV_DIR} \
        --val_ratio 0.15 \
        --source dermnet
    print("✅ 완료")

# 결과 확인
meta_path = f"{EXT_CSV_DIR}/metadata.json"
if Path(meta_path).exists():
    with open(meta_path) as _f:
        meta = _json.load(_f)
    print("\n클래스 분포 (train):")
    for cls, cnt in sorted(meta["class_distribution"]["train"].items()):
        print(f"  {cls:<12}: {cnt:>5}장")
    splits = meta["splits"]
    print(f"\n  train {splits.get('train',0)} | val {splits.get('val',0)} | test(홀드아웃) {splits.get('test',0)}")

In [ ]:
# ── 셀 8: 혼합 학습 실행 ──────────────────────────────────────
# AI Hub 합성 9,600 + DermNet 실제 3,373 혼합
# WeightedRandomSampler: AI Hub weight=1.0, DermNet weight=1.5

!EXTRA_DATA_DIR=data/processed/dermnet \
 EXTERNAL_WEIGHT=1.5 \
 EXPERIMENT_NAME=ds14_mixed_dermnet \
 CHECKPOINT_DIR=ai/results/DS14_mixed \
 python -m ai.training.classifier.train \
     --backbone densenet121 \
     --num_epochs 50 \
     --batch_size 64 \
     --root_dir {PROJECT_ROOT}

In [ ]:
# ── 셀 9: 평가 1 — 혼합 val (합성 성능 유지 확인) ───────────────
# DS14 val.csv 기준 — DS14_mixed가 기존 92% 수준을 유지하는지 확인
print("=" * 60)
print("평가 1: DS14 합성 val (성능 유지 확인)")
print("=" * 60)
!python -m ai.testing.evaluate \
    --checkpoint ai/results/DS14_mixed/best.pth \
    --data_dir data/processed/DS14 \
    --output_dir ai/results/DS14_mixed/eval_aihub_val \
    --root_dir {PROJECT_ROOT}

In [ ]:
# ── 셀 10: 평가 2 — DermNet 홀드아웃 test (도메인 갭 측정) ──────
# test.csv는 학습에 전혀 사용되지 않은 실제 이미지 1,096장
# 건선·아토피피부염·여드름 3종만 포함
# 목표: Top-1 ≥ 75% (기존 25% → 75% 이상으로 개선)
print("=" * 60)
print("평가 2: DermNet test 홀드아웃 (도메인 갭 측정)")
print("기존 베이스라인(DS14 only): ~25% → 목표: ≥ 75%")
print("=" * 60)
!python -m ai.testing.evaluate \
    --checkpoint ai/results/DS14_mixed/best.pth \
    --data_dir data/processed/dermnet \
    --split test \
    --output_dir ai/results/DS14_mixed/eval_dermnet_test \
    --root_dir {PROJECT_ROOT}

In [ ]:
# ── 셀 11: Threshold 최적화 ──────────────────────────────────
!python -m ai.testing.threshold_opt \
    --checkpoint ai/results/DS14_mixed/best.pth \
    --root_dir {PROJECT_ROOT}

In [ ]:
# ── 셀 12: 체크포인트 Drive 저장 (Colab 전용) ────────────────────
# 런타임 종료 전 반드시 실행 — 저장하지 않으면 학습 결과 소실
if IS_COLAB:
    import shutil
    from pathlib import Path

    CKPT_SRC = f"{COLAB_ROOT}/ai/results/DS14_mixed"
    CKPT_DST = f"{DRIVE_ROOT}/ai/results/DS14_mixed"

    if not Path(CKPT_SRC).exists():
        print(f"❌ 체크포인트 없음: {CKPT_SRC}")
    else:
        shutil.copytree(CKPT_SRC, CKPT_DST, dirs_exist_ok=True)
        print(f"✅ Drive 저장 완료: {CKPT_DST}")

    # DermNet 전처리 CSV도 Drive에 보존
    EXT_CSV_SRC = f"{COLAB_ROOT}/data/processed/dermnet"
    EXT_CSV_DST = f"{DRIVE_ROOT}/data/processed/dermnet"
    if Path(EXT_CSV_SRC).exists() and not Path(EXT_CSV_DST).exists():
        shutil.copytree(EXT_CSV_SRC, EXT_CSV_DST)
        print(f"✅ DermNet CSV Drive 저장: {EXT_CSV_DST}")
else:
    print("로컬 환경 — 체크포인트 이미 ai/results/DS14_mixed/ 에 저장됨")